In [ ]:
import os
import random
import sys
from hand_model_lite import HandModelMJCFLite
import numpy as np
import transforms3d
import torch
import trimesh


In [ ]:
mesh_path = "meshdata"

use_visual_mesh = True

''' For Egorhand'''
data_path = "dataset/DIP-Flex_opened_kinematics"
hand_file = "model/DIP-Flex_opened_kinematics.xml"
joint_names = [
    "Joint_pinkie_abduction", "Joint_pinkie_PPflexion", "Joint_pinkie_DPflexion",
                                "Joint_index_abduction", "Joint_index_PPflexion", "Joint_index_DPflexion",
                                "Joint_thumb_rotation", "Joint_thumb_abduction", "Joint_thumb_PPflexion", "Joint_thumb_DPflexion"
]

translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']


In [ ]:
hand_model = HandModelMJCFLite(
    hand_file,
    "model/assets")


In [ ]:
grasp_code_list = []
for code in os.listdir(data_path):
    grasp_code_list.append(code[:-4])

print(grasp_code_list)

In [ ]:
grasp_code = random.choice(grasp_code_list)
grasp_data = np.load(
    os.path.join(data_path, grasp_code+".npy"), allow_pickle=True)
object_mesh_origin = trimesh.load(os.path.join(
    mesh_path, grasp_code, "coacd/decomposed.obj"))
print(grasp_code)

print(grasp_data)
print(len(grasp_data))

In [ ]:
index = random.randint(0, len(grasp_data) - 1)
# index = 9


qpos = grasp_data[index]['qpos']
print(index)
rot = np.array(transforms3d.euler.euler2mat(
    *[qpos[name] for name in rot_names]))
rot = rot[:, :2].T.ravel().tolist()
hand_pose = torch.tensor([qpos[name] for name in translation_names] + rot + [qpos[name]
                         for name in joint_names], dtype=torch.float, device="cpu").unsqueeze(0)
hand_model.set_parameters(hand_pose)
hand_mesh = hand_model.get_trimesh_data(0)
object_mesh = object_mesh_origin.copy().apply_scale(grasp_data[index]["scale"])

# Задаем цвета (RGB в формате [R, G, B, A], где значения от 0.0 до 1.0)
hand_color = [0.7, 0.7, 0.7, 1.0]  # Красноватый цвет для руки
object_color = [0.2, 0.5, 0.8, 1.0]  # Голубоватый цвет для объекта

# Применяем цвета к мешам
hand_mesh.visual.face_colors = hand_color
object_mesh.visual.face_colors = object_color


In [ ]:
(hand_mesh+object_mesh).show()
